# DATCROSS — Scores de Transferência de AVC, ITU e Pneumonia

Este notebook implementa os três scores acadêmicos do projeto DATCROSS para **transferência inter-hospitalar**.

**Fórmula:**

\[
Score\ final = \frac{pontos\ obtidos}{pontos\ aplicáveis} \times 100
\]

**Classificação:**
- **70 a 100:** estável — elegível para transferência programada, após avaliação clínica
- **40 a 69:** risco intermediário — transferência condicionada à avaliação médica e definição do suporte
- **Abaixo de 40:** alta complexidade/instabilidade — transferência para serviço de maior complexidade, com avaliação de suporte avançado

Quando uma variável for `None`, ela é tratada como **NA** e sai do denominador.

> Observação: estes scores são adaptações acadêmicas para o projeto DATCROSS e não constituem instrumentos clínicos validados.

In [ ]:
def classificar_score(score, alerta_critico=False):
    if alerta_critico:
        return "ALTA COMPLEXIDADE / INSTABILIDADE"
    elif score >= 70:
        return "ESTÁVEL — transferência programada"
    elif score >= 40:
        return "RISCO INTERMEDIÁRIO — transferência condicionada"
    else:
        return "ALTA COMPLEXIDADE / INSTABILIDADE"

## 1. Score AVC

No AVC, o maior peso é dado ao estado neurológico, seguido da estabilidade respiratória e hemodinâmica.

**Regra crítica de transferência:** Glasgow 3–8 ou paciente inconsciente gera alerta de alta complexidade e avaliação para transporte em ambulância Tipo D — suporte avançado.

In [ ]:
def score_avc(
    glasgow,
    pas,
    fr,
    spo2,
    fc,
    temperatura,
    diurese,
    glicemia,
    respiracao_espontanea,
    oxigenio_suplementar,
    dispositivos,
    avc_estavel
):
    pontos = 0
    pontos_aplicaveis = 0

    # Glasgow — máximo 25
    if glasgow is not None:
        pontos_aplicaveis += 25
        if 13 <= glasgow <= 15:
            pontos += 25
        elif 9 <= glasgow <= 12:
            pontos += 12

    # PAS — máximo 10
    if pas is not None:
        pontos_aplicaveis += 10
        if 100 <= pas <= 139:
            pontos += 10
        elif 90 <= pas <= 99 or 140 <= pas <= 179:
            pontos += 5

    # FR — máximo 10
    if fr is not None:
        pontos_aplicaveis += 10
        if 12 <= fr <= 20:
            pontos += 10
        elif 21 <= fr <= 24:
            pontos += 5

    # SpO2 — máximo 10
    if spo2 is not None:
        pontos_aplicaveis += 10
        if spo2 >= 95:
            pontos += 10
        elif 92 <= spo2 <= 94:
            pontos += 5

    # FC — máximo 5
    if fc is not None:
        pontos_aplicaveis += 5
        if 60 <= fc <= 100:
            pontos += 5
        elif 101 <= fc <= 110:
            pontos += 2

    # Temperatura — máximo 5
    if temperatura is not None:
        pontos_aplicaveis += 5
        if 36 <= temperatura <= 37.2:
            pontos += 5
        elif 37.3 <= temperatura <= 37.9:
            pontos += 2

    # Diurese — máximo 5
    if diurese is not None:
        pontos_aplicaveis += 5
        if diurese >= 0.5:
            pontos += 5
        elif 0.3 <= diurese < 0.5:
            pontos += 2

    # Glicemia — máximo 5
    if glicemia is not None:
        pontos_aplicaveis += 5
        if 70 <= glicemia <= 140:
            pontos += 5
        elif 141 <= glicemia <= 180:
            pontos += 2

    # Respiração espontânea — máximo 10
    if respiracao_espontanea is not None:
        pontos_aplicaveis += 10
        if respiracao_espontanea:
            pontos += 10

    # Oxigênio suplementar — máximo 5
    if oxigenio_suplementar is not None:
        pontos_aplicaveis += 5
        if not oxigenio_suplementar:
            pontos += 5

    # Dispositivos — máximo 5
    if dispositivos is not None:
        pontos_aplicaveis += 5
        if dispositivos == 0:
            pontos += 5
        elif dispositivos == 1:
            pontos += 2

    # Imagem relacionada ao AVC — máximo 5
    if avc_estavel is not None:
        pontos_aplicaveis += 5
        if avc_estavel:
            pontos += 5

    score = (pontos / pontos_aplicaveis) * 100 if pontos_aplicaveis else 0

    alerta_critico = (
        glasgow is not None and 3 <= glasgow <= 8
    )

    return {
        "score": round(score, 1),
        "classificacao": classificar_score(score, alerta_critico),
        "pontos_obtidos": pontos,
        "pontos_aplicaveis": pontos_aplicaveis,
        "alerta_critico": alerta_critico
    }

In [ ]:
# 3 exemplos de AVC

avc_estavel = score_avc(
    glasgow=15, pas=120, fr=18, spo2=97, fc=80,
    temperatura=36.7, diurese=0.8, glicemia=110,
    respiracao_espontanea=True, oxigenio_suplementar=False,
    dispositivos=0, avc_estavel=True
)

avc_intermediario = score_avc(
    glasgow=10, pas=95, fr=22, spo2=93, fc=105,
    temperatura=37.5, diurese=0.4, glicemia=160,
    respiracao_espontanea=True, oxigenio_suplementar=True,
    dispositivos=1, avc_estavel=True
)

avc_complexidade = score_avc(
    glasgow=7, pas=85, fr=28, spo2=89, fc=120,
    temperatura=38.5, diurese=0.2, glicemia=280,
    respiracao_espontanea=False, oxigenio_suplementar=True,
    dispositivos=3, avc_estavel=False
)

print("AVC — Estável:", avc_estavel)
print("AVC — Risco intermediário:", avc_intermediario)
print("AVC — Alta complexidade:", avc_complexidade)

## 2. Score ITU

Na ITU, o score prioriza sinais vitais, resposta ao tratamento, urocultura, fatores urinários e estabilidade respiratória.

**Regra crítica de transferência:** múltiplos parâmetros alterados + urocultura positiva + antibioticoterapia em curso + critérios clínicos para UTI indicam transferência para serviço terciário/quaternário e avaliação para ambulância Tipo D.

In [ ]:
def score_itu(
    pas,
    glasgow,
    fr,
    spo2,
    fc,
    temperatura,
    diurese,
    glicemia,
    urocultura_tratada,
    horas_antibiotico,
    sonda_vesical,
    suporte_respiratorio,
    fatores_urinarios,
    outros_dispositivos,
    criterios_uti=False
):
    pontos = 0
    pontos_aplicaveis = 0
    parametros_alterados = 0

    if pas is not None:
        pontos_aplicaveis += 6
        if 100 <= pas <= 139:
            pontos += 6
        elif 90 <= pas <= 99:
            pontos += 3
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if glasgow is not None:
        pontos_aplicaveis += 6
        if 13 <= glasgow <= 15:
            pontos += 6
        elif 9 <= glasgow <= 12:
            pontos += 3
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if fr is not None:
        pontos_aplicaveis += 5
        if 12 <= fr <= 20:
            pontos += 5
        elif 21 <= fr <= 24:
            pontos += 2
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if spo2 is not None:
        pontos_aplicaveis += 5
        if spo2 >= 95:
            pontos += 5
        elif 92 <= spo2 <= 94:
            pontos += 2
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if fc is not None:
        pontos_aplicaveis += 4
        if 60 <= fc <= 100:
            pontos += 4
        elif 101 <= fc <= 110:
            pontos += 2
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if temperatura is not None:
        pontos_aplicaveis += 4
        if 36 <= temperatura <= 37.2:
            pontos += 4
        elif 37.3 <= temperatura <= 37.9:
            pontos += 2
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if diurese is not None:
        pontos_aplicaveis += 3
        if diurese >= 0.5:
            pontos += 3
        elif 0.3 <= diurese < 0.5:
            pontos += 2
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if glicemia is not None:
        pontos_aplicaveis += 2
        if 70 <= glicemia <= 140:
            pontos += 2
        elif 141 <= glicemia <= 180:
            pontos += 1
            parametros_alterados += 1
        else:
            parametros_alterados += 1

    if urocultura_tratada is not None:
        pontos_aplicaveis += 15
        if urocultura_tratada:
            pontos += 15

    if horas_antibiotico is not None:
        pontos_aplicaveis += 15
        if horas_antibiotico >= 72:
            pontos += 15
        elif 48 <= horas_antibiotico < 72:
            pontos += 8
        elif 0 < horas_antibiotico < 48:
            pontos += 3

    if sonda_vesical is not None:
        pontos_aplicaveis += 10
        if not sonda_vesical:
            pontos += 10

    if suporte_respiratorio is not None:
        pontos_aplicaveis += 10
        if suporte_respiratorio == "sem suporte":
            pontos += 10
        elif suporte_respiratorio == "o2":
            pontos += 5

    if fatores_urinarios is not None:
        pontos_aplicaveis += 10
        if fatores_urinarios == "nenhum":
            pontos += 10
        elif fatores_urinarios == "um":
            pontos += 5

    if outros_dispositivos is not None:
        pontos_aplicaveis += 5
        if outros_dispositivos == 0:
            pontos += 5

    score = (pontos / pontos_aplicaveis) * 100 if pontos_aplicaveis else 0

    alerta_critico = (
        parametros_alterados >= 2
        and urocultura_tratada is True
        and horas_antibiotico is not None
        and horas_antibiotico > 0
        and criterios_uti
    )

    return {
        "score": round(score, 1),
        "classificacao": classificar_score(score, alerta_critico),
        "pontos_obtidos": pontos,
        "pontos_aplicaveis": pontos_aplicaveis,
        "alerta_critico": alerta_critico
    }

In [ ]:
# 3 exemplos de ITU

itu_estavel = score_itu(
    pas=120, glasgow=15, fr=18, spo2=97, fc=80,
    temperatura=36.5, diurese=0.8, glicemia=110,
    urocultura_tratada=True, horas_antibiotico=80,
    sonda_vesical=False, suporte_respiratorio="sem suporte",
    fatores_urinarios="nenhum", outros_dispositivos=0,
    criterios_uti=False
)

itu_intermediario = score_itu(
    pas=95, glasgow=10, fr=22, spo2=93, fc=105,
    temperatura=37.5, diurese=0.4, glicemia=160,
    urocultura_tratada=True, horas_antibiotico=60,
    sonda_vesical=False, suporte_respiratorio="o2",
    fatores_urinarios="um", outros_dispositivos=1,
    criterios_uti=False
)

itu_complexidade = score_itu(
    pas=85, glasgow=7, fr=28, spo2=89, fc=125,
    temperatura=39, diurese=0.2, glicemia=260,
    urocultura_tratada=True, horas_antibiotico=24,
    sonda_vesical=True, suporte_respiratorio="iot_vm",
    fatores_urinarios="complicacao", outros_dispositivos=2,
    criterios_uti=True
)

print("ITU — Estável:", itu_estavel)
print("ITU — Risco intermediário:", itu_intermediario)
print("ITU — Alta complexidade:", itu_complexidade)

## 3. Score Pneumonia

Na pneumonia, oxigenação, frequência respiratória e necessidade de suporte respiratório recebem maior peso.

**Regra crítica de transferência:** pneumonia em ventilação mecânica com parâmetros ventilatórios/oxigenação desfavoráveis indica transferência para serviço terciário/quaternário e avaliação para ambulância Tipo D.

In [ ]:
def score_pneumonia(
    spo2,
    fr,
    suporte_respiratorio,
    temperatura,
    pas,
    glasgow,
    fc,
    horas_antibiotico,
    etiologia_definida,
    diurese
):
    pontos = 0
    pontos_aplicaveis = 0

    if spo2 is not None:
        pontos_aplicaveis += 15
        if spo2 >= 95:
            pontos += 15
        elif 92 <= spo2 <= 94:
            pontos += 7

    if fr is not None:
        pontos_aplicaveis += 15
        if 12 <= fr <= 20:
            pontos += 15
        elif 21 <= fr <= 24:
            pontos += 7

    if suporte_respiratorio is not None:
        pontos_aplicaveis += 15
        if suporte_respiratorio == "sem suporte":
            pontos += 15
        elif suporte_respiratorio == "o2":
            pontos += 7

    if temperatura is not None:
        pontos_aplicaveis += 10
        if 36 <= temperatura <= 37.2:
            pontos += 10
        elif 37.3 <= temperatura <= 37.9:
            pontos += 5

    if pas is not None:
        pontos_aplicaveis += 10
        if 100 <= pas <= 139:
            pontos += 10
        elif 90 <= pas <= 99:
            pontos += 5

    if glasgow is not None:
        pontos_aplicaveis += 10
        if 13 <= glasgow <= 15:
            pontos += 10
        elif 9 <= glasgow <= 12:
            pontos += 5

    if fc is not None:
        pontos_aplicaveis += 5
        if 60 <= fc <= 100:
            pontos += 5
        elif 101 <= fc <= 110:
            pontos += 2

    if horas_antibiotico is not None:
        pontos_aplicaveis += 10
        if horas_antibiotico >= 72:
            pontos += 10
        elif 48 <= horas_antibiotico < 72:
            pontos += 5
        elif 0 < horas_antibiotico < 48:
            pontos += 2

    if etiologia_definida is not None:
        pontos_aplicaveis += 5
        if etiologia_definida:
            pontos += 5
        else:
            pontos += 2

    if diurese is not None:
        pontos_aplicaveis += 5
        if diurese >= 0.5:
            pontos += 5
        elif 0.3 <= diurese < 0.5:
            pontos += 2

    score = (pontos / pontos_aplicaveis) * 100 if pontos_aplicaveis else 0

    alerta_critico = (
        suporte_respiratorio == "iot_vm"
        and (
            (spo2 is not None and spo2 < 92)
            or (fr is not None and fr >= 25)
        )
    )

    return {
        "score": round(score, 1),
        "classificacao": classificar_score(score, alerta_critico),
        "pontos_obtidos": pontos,
        "pontos_aplicaveis": pontos_aplicaveis,
        "alerta_critico": alerta_critico
    }

In [ ]:
# 3 exemplos de Pneumonia

pneumonia_estavel = score_pneumonia(
    spo2=97, fr=18, suporte_respiratorio="sem suporte",
    temperatura=36.6, pas=120, glasgow=15, fc=80,
    horas_antibiotico=80, etiologia_definida=True,
    diurese=0.8
)

pneumonia_intermediario = score_pneumonia(
    spo2=93, fr=22, suporte_respiratorio="o2",
    temperatura=37.5, pas=95, glasgow=10, fc=105,
    horas_antibiotico=60, etiologia_definida=False,
    diurese=0.4
)

pneumonia_complexidade = score_pneumonia(
    spo2=88, fr=30, suporte_respiratorio="iot_vm",
    temperatura=39, pas=85, glasgow=7, fc=125,
    horas_antibiotico=20, etiologia_definida=False,
    diurese=0.2
)

print("Pneumonia — Estável:", pneumonia_estavel)
print("Pneumonia — Risco intermediário:", pneumonia_intermediario)
print("Pneumonia — Alta complexidade:", pneumonia_complexidade)

## 4. Resumo dos 9 pacientes de teste

In [ ]:
import pandas as pd

resumo = pd.DataFrame([
    ["AVC", "Paciente 1", avc_estavel["score"], avc_estavel["classificacao"]],
    ["AVC", "Paciente 2", avc_intermediario["score"], avc_intermediario["classificacao"]],
    ["AVC", "Paciente 3", avc_complexidade["score"], avc_complexidade["classificacao"]],
    ["ITU", "Paciente 1", itu_estavel["score"], itu_estavel["classificacao"]],
    ["ITU", "Paciente 2", itu_intermediario["score"], itu_intermediario["classificacao"]],
    ["ITU", "Paciente 3", itu_complexidade["score"], itu_complexidade["classificacao"]],
    ["Pneumonia", "Paciente 1", pneumonia_estavel["score"], pneumonia_estavel["classificacao"]],
    ["Pneumonia", "Paciente 2", pneumonia_intermediario["score"], pneumonia_intermediario["classificacao"]],
    ["Pneumonia", "Paciente 3", pneumonia_complexidade["score"], pneumonia_complexidade["classificacao"]],
], columns=["Doença", "Paciente", "Score", "Classificação"])

resumo

## 5. Como testar um novo paciente

Basta chamar a função correspondente à doença e substituir os valores pelos dados do paciente.

Exemplo:

```python
novo_paciente = score_avc(
    glasgow=15,
    pas=125,
    fr=18,
    spo2=96,
    fc=82,
    temperatura=36.8,
    diurese=0.6,
    glicemia=115,
    respiracao_espontanea=True,
    oxigenio_suplementar=False,
    dispositivos=0,
    avc_estavel=True
)

novo_paciente
```